In [7]:
# Mount Google Drive
from google.colab import drive
import os
import shutil

# Try to unmount if already mounted to ensure a clean state
try:
  drive.flush_and_unmount()
  print("Drive unmounted successfully.")
except ValueError:
  print("Drive was not mounted or could not be unmounted.")

# Aggressive cleanup of the mount point using shell commands
# This ensures any stubborn files or incorrect permissions are removed.
mount_point = "/content/drive"
print(f"Performing aggressive cleanup of {mount_point}...")
!rm -rf "{mount_point}"
!mkdir -p "{mount_point}"
print(f"Cleanup complete. Mount point {mount_point} is now an empty directory.")

drive.mount(mount_point, force_remount=True)

from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

# ---------------- SETTINGS ----------------

SPLIT_ZIP = Path(
    "/content/drive/MyDrive/BreastDM_Project/data/segmentation_DS_7_14_2026.zip"
)

EXTRACT_DIR = Path("/content/BreastDM_split")
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 8

# ---------------- EXTRACT DATA ----------------

if not SPLIT_ZIP.exists():
    raise FileNotFoundError(f"Could not find: {SPLIT_ZIP}")

if not EXTRACT_DIR.exists():
    EXTRACT_DIR.mkdir(parents=True)
    shutil.unpack_archive(str(SPLIT_ZIP), str(EXTRACT_DIR))

# Locate the folder containing train/images
possible_roots = [
    EXTRACT_DIR,
    EXTRACT_DIR / SPLIT_ZIP.stem, # Corrected: Use SPLIT_ZIP.stem for the folder name
]

DATASET_ROOT = next(
    (folder for folder in possible_roots if (folder / "train" / "images").exists()),
    None,
)

if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Could not find train/images after extracting the split ZIP."
    )

print("Dataset location:", DATASET_ROOT)

# ---------------- DATASET CLASS ----------------

class BreastDMSegmentationDataset(Dataset):
    def __init__(self, split_directory, image_size=(256, 256)):
        self.image_directory = Path(split_directory) / "images"
        self.mask_directory = Path(split_directory) / "masks"
        self.image_size = image_size

        image_files = {
            path.stem: path
            for path in self.image_directory.iterdir()
            if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
        }

        mask_files = {
            path.stem: path
            for path in self.mask_directory.iterdir()
            if path.suffix.lower() == ".png"
        }

        missing_masks = image_files.keys() - mask_files.keys()
        missing_images = mask_files.keys() - image_files.keys()

        if missing_masks or missing_images:
            raise ValueError(
                f"Unmatched files found: "
                f"{len(missing_masks)} images without masks and "
                f"{len(missing_images)} masks without images."
            )

        self.pairs = [
            (image_files[name], mask_files[name])
            for name in sorted(image_files)
        ]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]

        # Convert the MRI image and mask to one channel
        image = Image.open(image_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        # Bilinear interpolation is appropriate for images
        image = TF.resize(
            image,
            self.image_size,
            interpolation=InterpolationMode.BILINEAR,
        )

        # Nearest-neighbor preserves the mask labels
        mask = TF.resize(
            mask,
            self.image_size,
            interpolation=InterpolationMode.NEAREST,
        )

        # Convert image values to the range 0–1
        image = TF.to_tensor(image)

        # Convert the mask into binary values: background=0, tumor=1
        mask = (TF.to_tensor(mask) > 0).float()

        return image, mask

# ---------------- CREATE DATALOADERS ----------------

train_dataset = BreastDMSegmentationDataset(
    DATASET_ROOT / "train", IMAGE_SIZE
)

val_dataset = BreastDMSegmentationDataset(
    DATASET_ROOT / "val", IMAGE_SIZE
)

test_dataset = BreastDMSegmentationDataset(
    DATASET_ROOT / "test", IMAGE_SIZE
)

loader_settings = {
    "batch_size": BATCH_SIZE,
    "num_workers": 2,
    "pin_memory": torch.cuda.is_available(),
}

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_settings,
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **loader_settings,
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **loader_settings,
)

# ---------------- QUICK TEST ----------------

images, masks = next(iter(train_loader))

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Testing examples:", len(test_dataset))
print("Image batch shape:", images.shape)
print("Mask batch shape:", masks.shape)
print("Mask values:", torch.unique(masks))


Drive not mounted, so nothing to flush and unmount.
Drive unmounted successfully.
Performing aggressive cleanup of /content/drive...
Cleanup complete. Mount point /content/drive is now an empty directory.
Mounted at /content/drive
Dataset location: /content/BreastDM_split/segmentation_DS_7_14_2026
Training examples: 20417
Validation examples: 4573
Testing examples: 4284
Image batch shape: torch.Size([8, 1, 256, 256])
Mask batch shape: torch.Size([8, 1, 256, 256])
Mask values: tensor([0., 1.])


In [8]:
print('Contents of EXTRACT_DIR after extraction:')
!ls -R {EXTRACT_DIR}


Streaming output truncated to the last 5000 lines.
BreaDM-Ma-1917__SUB8__p-043.png        BreaDM-Ma-2134__VIBRANT+C7__p-045.png
BreaDM-Ma-1917__SUB8__p-044.png        BreaDM-Ma-2134__VIBRANT+C7__p-046.png
BreaDM-Ma-1917__SUB8__p-045.png        BreaDM-Ma-2134__VIBRANT+C7__p-047.png
BreaDM-Ma-1917__SUB8__p-046.png        BreaDM-Ma-2134__VIBRANT+C7__p-048.png
BreaDM-Ma-1917__SUB8__p-047.png        BreaDM-Ma-2134__VIBRANT+C7__p-049.png
BreaDM-Ma-1917__SUB8__p-048.png        BreaDM-Ma-2134__VIBRANT+C7__p-050.png
BreaDM-Ma-1917__SUB8__p-049.png        BreaDM-Ma-2134__VIBRANT+C7__p-051.png
BreaDM-Ma-1917__VIBRANT+C1__p-043.png  BreaDM-Ma-2134__VIBRANT+C8__p-042.png
BreaDM-Ma-1917__VIBRANT+C1__p-044.png  BreaDM-Ma-2134__VIBRANT+C8__p-043.png
BreaDM-Ma-1917__VIBRANT+C1__p-045.png  BreaDM-Ma-2134__VIBRANT+C8__p-044.png
BreaDM-Ma-1917__VIBRANT+C1__p-046.png  BreaDM-Ma-2134__VIBRANT+C8__p-045.png
BreaDM-Ma-1917__VIBRANT+C1__p-047.png  BreaDM-Ma-2134__VIBRANT+C8__p-046.png
BreaDM-Ma-1917__VIBRANT+C

In [9]:
# Simple 2 Layer arch
# this is really basic
import torch
import torch.nn as nn

# two convolutional layers

class Unet_arch(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
        nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size = 3,
            padding = 1,
            bias = False, # Corrected: 'bais' to 'bias'
        ),
        nn.BatchNorm2d(out_channels), # Corrected: 'output_channels' to 'out_channels'
        nn.ReLU(inplace = True),

        # Second Layer!!
        nn.Conv2d(
            out_channels, # Corrected: 'output_channels' to 'out_channels'
            out_channels, # Corrected: 'output_channels' to 'out_channels'
            kernel_size = 3,
            padding = 1,
            bias = False, # Corrected: 'bais' to 'bias'
        ),
        nn.BatchNorm2d(out_channels), # Corrected: 'output_channels' to 'out_channels'
        nn.ReLU(inplace = True),
    )
    # now run it forward!!
    def forward(self, x):
        return self.layers(x)


In [10]:
# simple unet
# notice that it is a little longer

class Unet(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        # the Encoder
        self.encoder1 = Unet_arch(in_channels,32)
        self.encoder2 = Unet_arch(32,64)
        self.encoder3 = Unet_arch(64,128)
        self.encoder4 = Unet_arch(128,256)

        # pool it
        self.pool = nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)

        # bottom of Unet

        self.bottleneck = Unet_arch(256,512)


        # The Decoder

        self.upconv4 = nn.ConvTranspose2d(512,256,2,2)
        self.decoder4 = Unet_arch(512,256)

        self.upconv3 = nn.ConvTranspose2d(256,128,2,2)
        self.decoder3 = Unet_arch(256,128)

        self.upconv2 = nn.ConvTranspose2d(128,64,2,2)
        self.decoder2 = Unet_arch(128,64)

        self.upconv1 = nn.ConvTranspose2d(64,32,2,2)
        self.decoder1 = Unet_arch(64,32)

        # one output channel for the binary tumor mask

        self.output = nn.Conv2d(32, out_channels, kernel_size = 1)
        # forward function

    def forward(self,x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool(enc1))
        enc3 = self.encoder3(self.pool(enc2))
        enc4 = self.encoder4(self.pool(enc3))

        #bottleneck

        bottleneck = self.bottleneck(self.pool(enc4))

        #decoder with skip connections

        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4,enc4), dim = 1)
        dec4 = self.decoder4(dec4)

        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3,enc3), dim = 1)
        dec3= self.decoder3(dec3)

        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2,enc2), dim = 1)
        dec2 = self.decoder2(dec2)

        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1,enc1), dim = 1)
        dec1 = self.decoder1(dec1)

        # return it!!

        return self.output(dec1)


In [11]:
# forward function

def forward(self,x):
    enc1 = self.encoder1(x)
    enc2 = self.encoder2(self.pool(enc1))
    enc3 = self.encoder3(self.pool(enc2))
    enc4 = self.encoder4(self.pool(enc3))

    #bottleneck

    bottleneck = self.bottleneck(self.pool(enc4))

    #decoder with skip connections

    dec4 = self.upconv4(bottleneck)
    dec4 = torch.cat((dec4,enc4), dim = 1)
    dec4 = self.decoder4(dec4)

    dec3 = self.upconv3(dec4)
    dec3 = torch.cat((dec3,enc3), dim = 1)
    dec3= self.decoder3(dec3)

    dec2 = self.upconv2(dec3)
    dec2 = torch.cat((dec2,enc2), dim = 1)
    dec2 = self.decoder2(dec2)

    dec1 = self.upconv1(dec2)
    dec1 = torch.cat((dec1,enc1), dim = 1)
    dec1 = self.decoder1(dec1)

    # return it!!

    return self.output(dec1)

In [12]:
# Create and test it

# device + define model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Unet(1,1).to(device)

images, mask = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():
    predictions = model(images)

print("Device:", device)
print("Input shape:", images.shape)
print("Prediction shape:", predictions.shape)
print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
)


Device: cuda
Input shape: torch.Size([8, 1, 256, 256])
Prediction shape: torch.Size([8, 1, 256, 256])
Trainable parameters: 7,762,465


In [ ]:
# Training
from pathlib import Path
from tqdm.auto import tqdm
import torch
import torch.nn as nn

# ------------------------------------------------------------
# TRAINING SETTINGS
# ------------------------------------------------------------

NUM_EPOCHS = 10
LEARNING_RATE = 1e-4

MODEL_PATH = Path(
    "/content/drive/MyDrive/BreastDM_Project/models/best_unet.pth"
)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

bce_loss = nn.BCEWithLogitsLoss()


# ------------------------------------------------------------
# DICE LOSS AND DICE SCORE
# ------------------------------------------------------------

def dice_loss(logits, masks, smooth=1e-6):
    probabilities = torch.sigmoid(logits)

    probabilities = probabilities.flatten(start_dim=1)
    masks = masks.flatten(start_dim=1)

    intersection = (probabilities * masks).sum(dim=1)
    total = probabilities.sum(dim=1) + masks.sum(dim=1)

    dice = (2 * intersection + smooth) / (total + smooth)

    return 1 - dice.mean()


def dice_score(logits, masks, smooth=1e-6):
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities > 0.5).float()

    predictions = predictions.flatten(start_dim=1)
    masks = masks.flatten(start_dim=1)

    intersection = (predictions * masks).sum(dim=1)
    total = predictions.sum(dim=1) + masks.sum(dim=1)

    dice = (2 * intersection + smooth) / (total + smooth)

    return dice.mean()


def combined_loss(logits, masks):
    return bce_loss(logits, masks) + dice_loss(logits, masks)


# ------------------------------------------------------------
# TRAINING LOOP
# ------------------------------------------------------------

best_validation_dice = -1.0

history = {
    "train_loss": [],
    "train_dice": [],
    "val_loss": [],
    "val_dice": [],
}

for epoch in range(NUM_EPOCHS):

    # ---------------- TRAINING ----------------

    model.train()

    total_train_loss = 0
    total_train_dice = 0
    total_train_examples = 0

    training_progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} - Training",
    )

    for images, masks in training_progress:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = combined_loss(logits, masks)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)

        total_train_loss += loss.item() * batch_size
        total_train_dice += (
            dice_score(logits.detach(), masks).item() * batch_size
        )
        total_train_examples += batch_size

        training_progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = total_train_loss / total_train_examples
    train_dice = total_train_dice / total_train_examples

    # ---------------- VALIDATION ----------------

    model.eval()

    total_val_loss = 0
    total_val_dice = 0
    total_val_examples = 0

    with torch.no_grad():
        validation_progress = tqdm(
            val_loader,
            desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} - Validation",
        )

        for images, masks in validation_progress:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = combined_loss(logits, masks)

            batch_size = images.size(0)

            total_val_loss += loss.item() * batch_size
            total_val_dice += (
                dice_score(logits, masks).item() * batch_size
            )
            total_val_examples += batch_size

    val_loss = total_val_loss / total_val_examples
    val_dice = total_val_dice / total_val_examples

    # Save results for later plotting
    history["train_loss"].append(train_loss)
    history["train_dice"].append(train_dice)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)

    print(
        f"\nEpoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Train Dice: {train_dice:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f}"
    )

    # Save the model with the best validation Dice score
    if val_dice > best_validation_dice:
        best_validation_dice = val_dice

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch + 1,
                "validation_dice": val_dice,
            },
            MODEL_PATH,
        )

        print(f"Saved new best model to: {MODEL_PATH}")

print("\nTraining complete.")
print(f"Best validation Dice: {best_validation_dice:.4f}")

Epoch 1/10 - Training:   0%|          | 0/2553 [00:00<?, ?it/s]

Epoch 1/10 - Validation:   0%|          | 0/572 [00:00<?, ?it/s]


Epoch 1/10 | Train loss: 1.0385 | Train Dice: 0.6307 | Val loss: 0.8414 | Val Dice: 0.4413
Saved new best model to: /content/drive/MyDrive/BreastDM_Project/models/best_unet.pth


Epoch 2/10 - Training:   0%|          | 0/2553 [00:00<?, ?it/s]

Epoch 2/10 - Validation:   0%|          | 0/572 [00:00<?, ?it/s]

...


Epoch 7/10 | Train loss: 0.0925 | Train Dice: 0.9116 | Val loss: 0.4813 | Val Dice: 0.5306


Epoch 8/10 - Training:   0%|          | 0/2553 [00:00<?, ?it/s]

Epoch 8/10 - Validation:   0%|          | 0/572 [00:00<?, ?it/s]


Epoch 8/10 | Train loss: 0.0869 | Train Dice: 0.9169 | Val loss: 0.4904 | Val Dice: 0.5214


Epoch 9/10 - Training:   0%|          | 0/2553 [00:00<?, ?it/s]